# How Logistic Regression Works

**Notebook 3 of 6.** The threshold rule in notebook 02 gave every penguin a hard answer:
above 3,700 g male, below it female. A penguin of 3,690 g and one of 2,900 g got the same
answer, although one is a close call and the other is not.

**Logistic regression** gives a **probability** instead: how likely this penguin is to be
male. It is the standard first classifier, and this notebook takes it apart on the
smallest possible problem: **one species and one measurement**. The model turns out to be
just two numbers, and by the end you will compute its predictions by hand.

Notebook 02 showed why one species: the heavy Gentoo females outweigh most males of the
other species, so body mass mixes up sex and species. The 146 Adelie penguins with a
recorded sex keep the question clean.

In [ ]:
# data analysis
import numpy as np
import pandas as pd

# visualization
import matplotlib.pyplot as plt
import seaborn as sns

# machine learning
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

sns.set_style("whitegrid")

In [ ]:
df_penguins = pd.read_csv("data/penguins.csv")
adelie = df_penguins[(df_penguins["species"] == "Adelie") & df_penguins["sex"].notna()]

X = adelie[["body_mass_g"]]
y = (adelie["sex"] == "MALE").astype(int)

X.shape, y.value_counts()

Two details in that cell:

- `X` is selected with **double brackets**, `adelie[["body_mass_g"]]`. That keeps it a
  table with one column rather than a single column, because scikit-learn always expects
  the features as a table, even when there is only one.
- The target is recoded as **1 for male and 0 for female**. The model would accept the
  text labels too, but numbers make the plots and the arithmetic below readable. The class
  coded 1 is called the **positive class**. That is a label, not a judgment.

## 1. What the data looks like

With a 0/1 target, the data is two horizontal strips of dots. A little vertical jitter
keeps the dots from hiding each other.

In [ ]:
rng = np.random.default_rng(0)
jitter = rng.uniform(-0.04, 0.04, size=len(y))

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.scatter(X["body_mass_g"], y + jitter, c=y, cmap="coolwarm", alpha=0.6)
ax.set_yticks([0, 1], ["female (0)", "male (1)"])
ax.set_xlabel("body mass (g)")
ax.set_title("Adelie penguins: body mass and sex");

Every Adelie lighter than 3,325 g is female, and every one heavier than 3,900 g is male.
In between, both sexes occur, and that overlap holds 72 of the 146 penguins. What we want is a curve that is close to 0 on
the left, close to 1 on the right, and rises smoothly through the overlap, so that its
height at any mass reads as the **probability of male**.

## 2. The sigmoid

A straight line cannot do this: it runs below 0 and above 1, and a probability cannot. The
curve logistic regression uses instead is the **sigmoid**:

$$\sigma(t) = \frac{1}{1 + e^{-t}}$$

It takes any number `t` and squeezes it into the range between 0 and 1.

In [ ]:
def sigmoid(t):
    """Squeeze any number, or array of numbers, into the range 0 to 1.

    Parameters
    ----------
    t : float or numpy.ndarray
        The input, any real number.

    Returns
    -------
    float or numpy.ndarray
        1 / (1 + e^-t), with the same shape as `t`.
    """
    return 1 / (1 + np.exp(-t))


t = np.linspace(-8, 8, 200)

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(t, sigmoid(t))
ax.axhline(0.5, color="grey", linestyle=":")
ax.axvline(0, color="grey", linestyle=":")
ax.set_xlabel("t")
ax.set_ylabel("sigmoid(t)")
ax.set_title("The sigmoid squeezes every number into 0 to 1");

> **Exercise:**
>
> Compute `sigmoid` at `t = 0`, `t = 3` and `t = -3`. What does the sigmoid return at 0,
> and how are the values at 3 and at -3 related?

In [ ]:
# your code: the sigmoid at 0, 3 and -3

<details><summary>
Click here to compare your answer...
</summary>

`sigmoid(0)` is exactly **0.5**, the point of greatest uncertainty. `sigmoid(3)` is about
0.953 and `sigmoid(-3)` about 0.047: they add up to 1. The curve is symmetric around the
middle, so a value of `t` as far below 0 as another is above it is equally sure, just of
the other class.
</details>

## 3. From a measurement to a probability

The sigmoid turns `t` into a probability, but what is `t`? Logistic regression makes it a
**straight line in the feature**:

$$t = b_0 + b_1 \cdot x \qquad\qquad p(\text{male}) = \sigma(b_0 + b_1 \cdot x)$$

Here `x` is the body mass, `b_1` is the **coefficient** that says how fast the probability
rises with mass, and `b_0` is the **intercept** that shifts the curve left or right.

**The whole model is these two numbers.** Training it means finding the `b_0` and `b_1`
under which the labels in the training set are as likely as possible: males get high
probabilities, females low ones. You do not have to do that search yourself, that is what
`fit` is for.

## 4. Fitting the model

The split first, exactly as in notebook 02. With 146 penguins, a quarter is 37.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

model = LogisticRegression()
model.fit(X_train, y_train)

Every scikit-learn model follows this pattern: **create** it, then **fit** it on the
training features and target. What it learned is stored on the model, in attributes whose
names end in an underscore:

In [ ]:
b0 = model.intercept_[0]
b1 = model.coef_[0][0]

print(f"intercept b0:   {b0:.3f}")
print(f"coefficient b1: {b1:.5f} per gram")

The coefficient is **positive**: the heavier the penguin, the higher the probability of
male, as the plot suggested. It looks tiny because the unit is a single gram. Multiplied
by 100, it says that every 100 g adds about 0.7 to `t`.

## 5. A prediction by hand

`predict_proba` returns the model's probabilities, one column per class. The columns are
in the order of `model.classes_`, here 0 then 1, so the **second column** is the
probability of male.

In [ ]:
model.classes_

In [ ]:
model.predict_proba(X_test.head())

> **Exercise:**
>
> Compute by hand the probability that an Adelie of **4,000 g** is male: first `t` from
> `b0` and `b1`, then the `sigmoid`. Check your number against `model.predict_proba`.

<details><summary>
Click here for a hint...
</summary>

`predict_proba` needs a table with the same column name as the training features:
`pd.DataFrame({"body_mass_g": [4000]})`.
</details>

In [ ]:
# your code: the probability of male at 4,000 g, by hand and from the model

<details><summary>
Click here to compare your answer...
</summary>

Both give about **0.89**. There is nothing more inside the model: two numbers, a straight
line and the sigmoid. Everything else logistic regression does is built on this.
</details>

## 6. The decision boundary

`predict` turns a probability into a class, by predicting male wherever the probability is
above 0.5. The sigmoid is 0.5 exactly where `t` is 0, that is where
`b_0 + b_1 \cdot x = 0`:

$$x = -\frac{b_0}{b_1}$$

This mass is the **decision boundary**. Unlike the threshold in notebook 02, nobody chose
it: it follows from the fitted coefficients.

In [ ]:
boundary = -b0 / b1
print(f"decision boundary: {boundary:.0f} g")

# predict() is the same as a probability above 0.5
proba_male = model.predict_proba(X_test)[:, 1]
print("predict agrees with proba > 0.5:", (model.predict(X_test) == (proba_male > 0.5)).all())

All of it in one picture: the training penguins, the fitted curve, and the boundary.

In [ ]:
mass_grid = pd.DataFrame({"body_mass_g": np.linspace(2800, 4800, 200)})
jitter = rng.uniform(-0.04, 0.04, size=len(y_train))

fig, ax = plt.subplots(figsize=(8, 4))
ax.scatter(X_train["body_mass_g"], y_train + jitter, c=y_train, cmap="coolwarm", alpha=0.6)
ax.plot(mass_grid["body_mass_g"], model.predict_proba(mass_grid)[:, 1],
        color="black", label="probability of male")
ax.axhline(0.5, color="grey", linestyle=":")
ax.axvline(boundary, color="black", linestyle="--", label=f"boundary, {boundary:.0f} g")
ax.set_xlabel("body mass (g)")
ax.set_ylabel("probability of male")
ax.set_title("Logistic regression on Adelie body mass")
ax.legend(loc="center right");

Far from the boundary the curve is flat and the model is sure; near it the curve is steep
and the model hedges. That is what the hard threshold could not express.

Picking a cut-off other than 0.5, to catch more males at the cost of more mistakes, is a
choice with consequences of its own, and a topic of its own. Here the cut-off stays at 0.5.

Now the score that counts, on the test penguins, against the majority baseline:

In [ ]:
majority = y_train.mode()[0]

print(f"baseline, test:  {(y_test == majority).mean():.3f}")
print(f"model, train:    {accuracy_score(y_train, model.predict(X_train)):.3f}")
print(f"model, test:     {accuracy_score(y_test, model.predict(X_test)):.3f}")

Well above the baseline of about 49 percent. The test score is even higher than the
training score, which is luck of a small test set rather than a sign of anything: 37
penguins, so each one moves the score by almost three points.

> **Exercise:**
>
> From which body mass on is the model **at least 90 percent sure** that an Adelie is
> male? Find it either with algebra or by searching a grid of masses.

<details><summary>
Click here for a hint...
</summary>

With a grid: compute `model.predict_proba(mass_grid)[:, 1]` and find the first mass where
it reaches 0.9. With algebra: the sigmoid is 0.9 where `t = ln(0.9 / 0.1) = ln 9`, so solve
`b0 + b1 * x = np.log(9)` for `x`.
</details>

In [ ]:
# your code: the mass at which the probability of male reaches 0.9

<details><summary>
Click here to compare your answer...
</summary>

About **4,010 g**, a little over 300 g past the decision boundary. The grid gives the same
answer to within its spacing. Between the boundary and that mass, the model predicts male
but is not very sure of it.
</details>

## 7. A second species

> **Exercise:**
>
> Fit the same model, body mass against sex, on the **Gentoo** penguins, with the same
> split settings. Where is their decision boundary, and what does the comparison with the
> Adelie boundary say about fitting one body-mass model to all species at once?

In [ ]:
# your code: the same model on Gentoo penguins, and its decision boundary

<details><summary>
Click here to compare your answer...
</summary>

The Gentoo boundary lies at about **5,040 g**, some 1,340 g above the Adelie one. A
4,500 g penguin is almost certainly a male Adelie or a female Gentoo. One body-mass
model for all species would have to put its boundary somewhere in between and be wrong
for both.

The way out is not a model per species, it is to give the model **more than one feature**,
so that it can take the species, or measurements that reveal it, into account. That is the
next notebook.
</details>

## Summary

- Logistic regression predicts a **probability**, `sigmoid(b0 + b1 * x)`. The sigmoid
  squeezes a straight line into the range 0 to 1.
- The fitted model is just the **intercept** `intercept_` and the **coefficient** `coef_`.
  A positive coefficient means the probability rises with the feature.
- `predict_proba` gives one column per class, in the order of `classes_`. `predict` is a
  probability above 0.5.
- The **decision boundary** is where the probability is 0.5, at `x = -b0 / b1`. It comes
  out of the fit rather than being chosen.
- Every scikit-learn model follows the same pattern: **create, fit, predict**.

Next, in `04_logistic_regression_sklearn.ipynb`: all four measurements and all three
species in one model, and a target with three classes instead of two.